**Task 3: Functions for cleaning and analysing data
using NumPy**

1.

In [13]:
import numpy as np
import pandas as pd

In [14]:
def clean_price_outliers(prices: list[float] | np.ndarray) -> np.ndarray:
    """
    Replace price outliers with the median of valid prices.

    The function converts the input prices to a 1D NumPy float array,
    identifies values more than 3 standard deviations from the mean,
    and replaces detected outliers with the median of valid prices.

    Args:
        prices: A list or NumPy array containing prices.

    Returns:
        A 1D NumPy array with outliers replaced by the median.
    """

     # Convert input to a one-dimensional NumPy float array
    prices_array = np.asarray(prices, dtype=float).copy()

    if prices_array.ndim != 1:
        raise ValueError("Prices must be a 1D array.")

    if prices_array.size == 0:
        return prices_array

    # Calculate mean and standard deviation
    mean = np.mean(prices_array)
    std = np.std(prices_array)

    # Identify outliers more than 3 standard deviations from the mean
    outlier_mask = np.abs(prices_array - mean) > 3 * std

    # Find the median of valid prices and replace outliers
    valid_prices = prices_array[~outlier_mask]

    if valid_prices.size > 0:
        median = np.median(valid_prices)
        prices_array[outlier_mask] = median

    return prices_array


In [15]:
prices = np.array([4.50, 4.80, 5.00, 4.70, 999.00])

cleaned_prices = clean_price_outliers(prices)

print("Original prices:", prices)
print("Cleaned prices:", cleaned_prices)


Original prices: [  4.5   4.8   5.    4.7 999. ]
Cleaned prices: [  4.5   4.8   5.    4.7 999. ]


Testing and Observation

The test includes an extreme price of 999.00. The value is not replaced
in this small dataset because the outlier itself increases the mean and
standard deviation. As a result, it does not exceed the three-standard-
deviation threshold.

This demonstrates a limitation of the specified outlier detection method.

2.

In [17]:
def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray
) -> np.ndarray:
    """
    Compute the refund ratio for each item using NumPy.

    The ratio is calculated by dividing refund counts by sales
    counts plus a small epsilon to avoid division by zero.

    Args:
        sales_count: A NumPy array containing the number of sales.
        refund_count: A NumPy array containing the number of refunds.

    Returns:
        A NumPy array containing the refund ratios.
    """

    epsilon = 1e-6
    return refund_count / (sales_count + epsilon)


**Task 4: Data analysis using Pandas**

1.

In [18]:
def create_transactions_df(
    transactions: list[dict]
) -> pd.DataFrame:
    """
    Convert transaction records into a chronologically sorted DataFrame.

    Args:
        transactions: A list of dictionaries containing transaction records.

    Returns:
        A DataFrame with datetime timestamps as the sorted index.
    """
    df = pd.DataFrame(transactions)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df.sort_index()

    return df

2.

In [19]:
def resample_hourly_revenue(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Calculate total transaction price per item in hourly windows.

    Args:
        df: A DataFrame indexed by timestamp, containing item and price columns.

    Returns:
        A DataFrame with total price for each item in each hourly window.
    """
    hourly_revenue = (
        df.groupby("item")
        .resample("1h")["price"]
        .sum()
        .reset_index()
    )

    return hourly_revenue

3.

In [26]:
def create_status_pivot_table(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Create a pivot table showing total transaction revenue per item
    across SUCCESS, REFUND, and CANCELLED statuses.

    Args:
        df: A DataFrame containing item, price, and status columns.

    Returns:
        A pivot table with items as rows and status categories as columns.
    """
    status_pivot = pd.pivot_table(
        df,
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0
    )

    status_pivot = status_pivot.reindex(
        columns=["SUCCESS", "REFUND", "CANCELLED"],
        fill_value=0
    )

    return status_pivot

4.

In [21]:
def calculate_item_refund_ratios(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Calculate the refund ratio for each item using the NumPy function
    from Task 3.

    Args:
        df: A DataFrame containing item and status columns.

    Returns:
        A DataFrame containing each item, its sales count,
        refund count, and refund ratio.
    """
    sales_count = (
        df[df["status"] == "SUCCESS"]
        .groupby("item")
        .size()
    )

    refund_count = (
        df[df["status"] == "REFUND"]
        .groupby("item")
        .size()
    )

    counts = pd.concat(
        [sales_count, refund_count],
        axis=1
    ).fillna(0)

    counts.columns = ["sales_count", "refund_count"]

    counts["refund_ratio"] = compute_refund_ratio(
        counts["sales_count"].to_numpy(),
        counts["refund_count"].to_numpy()
    )

    return counts.reset_index()